# Predicting Basketball Shot Outcomes Using a Multilayer Perceptron
**APOAI2025 Mock Competition Question 1**

For this question, I used a PyTorch neural network to estimate whether a basketball shot would be made using only its court coordinates in the .csv file provided for the problem.

Since I've enjoyed this question the most, I also tried figuring out whether the other combinations of hyperparameters for the neural network's inputs would make the model better **for the love of the game** and because I'm a big fan of analyzing NBA's data which I've already done all the time way before. My conclusion regarding this problem is located at the end of the notebook.

# 1. Objective

The task is to create a binary classification whrein you predict `1` if a shot is made or `0` if it is a missed shot. I do this by checking the inputs of `loc_x` and `loc_y`. As mentioned, the MLP returns a make probability wherein I use a threshold of `0.5` to convert that prbability into a predicted label.

# 2. Experimental Setup

I only used my Macbook's CPU for training with one PyTorch thread and fixed some random seeds for a repeated experiment. The configuration is fixed before the actual evalutation of the holdout.


| Setting | Value | Basis |
| --- | ---: | --- |
| Training epochs | 270 | Asked an LLM for a suggested number for the training |
| Model initialization seed | 43 | Asked an LLM for a suggested number for the training  |
| Learning-rate schedule length | 400 | Asked an LLM for a suggested number for the training |
| Batch size | 1,024 | A manually chosen starting setting |
| Initial learning rate | 0.01 | A manually chosen starting setting |
| Weight decay | 0.0001 | A manually chosen small penalty on large weights |
| Split seed | 42 | A manually chosen fixed arbitrary seed for reproducible row selection. |

In [1]:
from pathlib import Path
from time import perf_counter

NOTEBOOK_START = perf_counter()


def find_question_folder(name, marker):
    cwd = Path.cwd().resolve()

    for base in (cwd, *cwd.parents):
        candidates = [base] if base.name == name else []
        candidates += [base / name, base / 'ICAIC_Outputs' / name]

        for folder in candidates:
            if (folder / marker).exists():
                return folder.resolve()
    if (cwd / marker).exists():
        return cwd

    raise FileNotFoundError(
        f'Cannot find {name}/{marker} from {cwd}. '
        'Keep the supplied data inside its question folder.'
    )

In [2]:
import csv
import math
import random
from pathlib import Path
import torch
from torch import nn
import sys

EPOCHS = 270
SCHEDULE_EPOCHS = 400
BATCH_SIZE = 1024
LEARNING_RATE = 0.01
SEED = 43
FOLDER = find_question_folder('Question1', 'data_train.csv')


print(f"Notebook folder: {FOLDER}")
print(f"CSV: {FOLDER / 'data_train.csv'}")

Notebook folder: /Users/davidfrancisco/Documents/ICAIC_Outputs/Question1
CSV: /Users/davidfrancisco/Documents/ICAIC_Outputs/Question1/data_train.csv


# 3 Read Dataset and Split Evaluation

The given `data_train.csv` contains 25,000 shots. I define "labeled shots" as those shots with either `0` or `1` dependig whether it got shot inside or not. I then split the labeled rows into 2 in which 20,000 is used for the local evaluation model and 5,000 is held aside to check its predictions.

Each row provides coordinates and a result of the shot. I excluded the three columns namely `minutes_remaining`, `shot_distance`, and `shot_id` from the inputs as the task only requires the coordinates.

The split is also stratified after where approximately 20% of the made shots and 20% of the missed shots are both held out seperately for the checking after so I can keep the outcome proportions similar across the two groups.

In [3]:
def read_data():
    features = []
    labels = []

    with open(FOLDER / "data_train.csv", newline="", encoding="utf-8-sig") as file:
        for row in csv.DictReader(file):
            features.append([float(row["loc_x"]), float(row["loc_y"])])
            labels.append([float(row["shot_made_flag"])])

    features = torch.tensor(features, dtype=torch.float32)
    labels = torch.tensor(labels, dtype=torch.float32)
    return features, labels

def split_data(labels):
    rng = random.Random(42)
    answers = labels.tolist()
    training = []
    holdout = []

    for label in (0, 1):
        group = []
        for index in range(len(answers)):
            if answers[index][0] == label:
                group.append(index)

        rng.shuffle(group)
        count = round(0.20 * len(group))
        holdout.extend(group[:count])
        training.extend(group[count:])

    rng.shuffle(training)
    rng.shuffle(holdout)
    return torch.tensor(training), torch.tensor(holdout)


In [4]:
torch.set_num_threads(1)
features, labels = read_data()
training, holdout = split_data(labels)
print(f"Labeled shots: {len(features)}")
print(f"Evaluation training rows: {len(training)}; holdout rows: {len(holdout)}")
print(f"Fixed settings: seed={SEED}, epochs={EPOCHS}, batch size={BATCH_SIZE}")

print(f"Made shots: {int(labels.sum())}")
print(f"Missed shots: {len(labels) - int(labels.sum())}")
print(f"Feature shape: {tuple(features.shape)}; label shape: {tuple(labels.shape)}")

Labeled shots: 25000
Evaluation training rows: 20000; holdout rows: 5000
Fixed settings: seed=43, epochs=270, batch size=1024
Made shots: 11186
Missed shots: 13814
Feature shape: (25000, 2); label shape: (25000, 1)


# 4. Neural Network's Architecture

I used two hidden layers with eight neurons each and one output neuron:

`[loc_x, loc_y]` -> Linear(2, 8) -> ReLU -> Linear(8, 8) -> ReLU -> Linear(8, 1) -> Sigmoid

ReLU changes the negative neurons outptus to zero which creates the bends which will show how the network responds to its inputs which hekps learns patterns that a straight line can not teach. The sigmoid at theend makes the final neuron score to be between 0 ad 1 which gives out an estamated make probability.

In [5]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 8)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(8, 8)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        x = self.sigmoid(x)
        return x

# 5. Coordinate Preprocessing

I then standardize each coordinate before training following the formula:

`scaled value = (original value - training mean) / training std`

For the evaluation experiment, the means and standard deviations are directly calculated from the 20,000 training rows only and that the holdout coordiates use those same statistics as well but does not influence the preprocessing. This scaling is performed outside the network first so before exporting the final parameters, I incorporate it into the first linear layer of the network I have on step 4.

In [6]:
training_features = features[training]
mean = training_features.mean(dim=0)
std = training_features.std(dim=0, unbiased=False)
training_features = (training_features - mean) / std
holdout_features = (features[holdout] - mean) / std

# 6. Training method

The training model utilizes the Class I made in the earlier Python cell wherein for every shuffled batch, the network predicts make probabilites and then the binary cross-entropy that I set compares them with the answers. After, Adam updates the weights and biases.


In [7]:
def train_model(features, labels):
    torch.manual_seed(SEED)
    model = Model()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=LEARNING_RATE, weight_decay=0.0001
    )
    loss_function = nn.BCELoss()
    generator = torch.Generator().manual_seed(SEED + 1000)

    for epoch in range(1, EPOCHS + 1):
        learning_rate = LEARNING_RATE * (
            0.1 + 0.9 * (1 + math.cos(math.pi * (epoch - 1) / SCHEDULE_EPOCHS)) / 2
        )
        optimizer.param_groups[0]["lr"] = learning_rate
        order = torch.randperm(len(features), generator=generator)

        for start in range(0, len(features), BATCH_SIZE):
            batch = order[start : start + BATCH_SIZE]
            optimizer.zero_grad()
            probabilities = model(features[batch])
            loss = loss_function(probabilities, labels[batch])
            loss.backward()
            optimizer.step()

        if epoch % 100 == 0 or epoch == EPOCHS:
            print(f"Epoch {epoch}/{EPOCHS}", flush=True)

    model.eval()
    return model

# 7. Holdout Evaluation

I assess the model without updating its weights where accuraty counts the fraction of predicted labels matching the actual answers while the binary cross-entropy also assess the probabilities. Moreover, the comparison baseline always predicts whichever outcome is more common in the 20,000 training rows I split in which the held-out answers are used to score that fixed rule.

In [8]:
def evaluate(model, features, labels):
    with torch.no_grad():
        probabilities = model(features)
        loss = nn.BCELoss()(probabilities, labels).item()
        predictions = (probabilities >= 0.5).float()
        correct = (predictions == labels).sum().item()

    return {"accuracy": correct / len(labels), "bce": loss, "correct": correct}

In [9]:
print(f"Training on {len(training)} rows for the holdout evaluation:")
evaluation_model = train_model(training_features, labels[training])
metrics = evaluate(evaluation_model, holdout_features, labels[holdout])
majority_label = float(labels[training].mean() >= 0.5)
baseline = (labels[holdout] == majority_label).float().mean().item()
print(f"Holdout results: {metrics}")
print(f"Holdout accuracy: {metrics['accuracy']:.2%}")
print(f"Always-majority baseline: {baseline:.2%}")

Training on 20000 rows for the holdout evaluation:
Epoch 100/270
Epoch 200/270
Epoch 270/270
Holdout results: {'accuracy': 0.6036, 'bce': 0.6648858189582825, 'correct': 3018}
Holdout accuracy: 60.36%
Always-majority baseline: 55.26%


### Results and interpretation

| Method | Correct predictions | Holdout accuracy | Binary cross-entropy |
| --- | ---: | ---: | ---: |
| Always predict the majority outcome (miss) | 2,763 / 5,000 | 55.26% | NA |
| Multilayer perceptron | 3,018 / 5,000 | 60.36% | 0.664886 |

The network correctly classified 255 more shots than the baseline which is an improvement of 5.10 percentage points. Because of this, I know that the model performs better than always guessing the more common answer. I therefore treat its probabilities as uncertain estimates first rather than actual guaranteed predicitons about individual shots.

# 8. Producing final trained model
After evaluating the initial approach of using 20,000 rows for training and 5,000 rows for testing, I then trained a fresh final model on all of the 25,000 labeled shots. This uses new scaling statistics that are calculated from its full training data and retains the same training settings.

In [10]:
mean = features.mean(dim=0)
std = features.std(dim=0, unbiased=False)
scaled_features = (features - mean) / std

print(f"Training on all {len(features)} rows for the final saved model:")
model = train_model(scaled_features, labels)

Training on all 25000 rows for the final saved model:
Epoch 100/270
Epoch 200/270
Epoch 270/270


## 9. Exporting the model parameters

Since the problem uses original coordinate values, I adjust the first layer so it performs the same calculation that it performed with scaled inputs since it will produce the wrong values otherwise:

```text
new weights = trained weights / standard deviation
new bias = trained bias - sum(new weights × mean)
```

In [11]:
exported_model = Model()
exported_model.load_state_dict(model.state_dict())
exported_model.eval()

with torch.no_grad():
    raw_weights = model.fc1.weight / std
    raw_bias = model.fc1.bias - (raw_weights * mean).sum(dim=1)
    exported_model.fc1.weight.copy_(raw_weights)
    exported_model.fc1.bias.copy_(raw_bias)

In [12]:
model.eval()
exported_model.eval()

with torch.no_grad():
    raw_inputs = features[:256]
    expected = model((raw_inputs - mean) / std)
    exported_predictions = exported_model(raw_inputs)

torch.testing.assert_close(
    exported_predictions,
    expected,
    rtol=1e-5,
    atol=1e-6,
)

print('Export check passed: raw-coordinate predictions match the scaled model.')

Export check passed: raw-coordinate predictions match the scaled model.


The cell below saves `submission_dic.pth` and `submission_model.py` in which the .pth file contains the exported weights and biases while the .py file contains the matching `Model` defintion.

In [13]:
output = FOLDER / "submission_dic.pth"
torch.save(exported_model.state_dict(), output)

model_definition = """from torch import nn

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 8)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(8, 8)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        x = self.sigmoid(x)
        return x
"""
(FOLDER / "submission_model.py").write_text(model_definition)

print(f"Model definition: {FOLDER / 'submission_model.py'}")
print(f"Trained parameters: {output}")


Model definition: /Users/davidfrancisco/Documents/ICAIC_Outputs/Question1/submission_model.py
Trained parameters: /Users/davidfrancisco/Documents/ICAIC_Outputs/Question1/submission_dic.pth


# 10. Reloading the training model and demonstrating its prediction

To demonstate its prediction, I reload the dictionary into a fresh model and supply original, unscaled coordinates. I picked two rows of the supplised CSV.

In [14]:
saved_model = Model()
saved_model.load_state_dict(torch.load(output, map_location="cpu", weights_only=True))
saved_model.eval()

positions = features[:2]
actual_labels = labels[:2].squeeze(1).int()

with torch.no_grad():
    probabilities = saved_model(positions).squeeze(1)
    predicted_labels = (probabilities >= 0.5).int()

print("Coordinates:", positions.tolist())
print("Estimated make probabilities:", probabilities.tolist())
print("Predicted labels:", predicted_labels.tolist())
print("Actual labels:", actual_labels.tolist())
print("Correct predictions:", (predicted_labels == actual_labels).tolist())

Coordinates: [[0.0, 0.0], [-3.0, 130.0]]
Estimated make probabilities: [0.6353702545166016, 0.485137403011322]
Predicted labels: [1, 0]
Actual labels: [1, 0]
Correct predictions: [True, True]


In [15]:
elapsed_seconds = perf_counter() - NOTEBOOK_START
print(f'Measured notebook runtime for this run: {elapsed_seconds:.2f} seconds')

Measured notebook runtime for this run: 5.01 seconds


# 11. Conclusion

I trained a multilayer perceptron using `loc_x` and `loc_y` while keeping the architecture within the required limits of three linear layers and at most eight neurons in each hidden layer. On the local holdout, the evaluation model correctly predicted **3,018 of 5,000 shots** which gave a total **60.36% accuracy** compared with **55.26%** from always predicting the more common outcome which was a missed shot.

This is an improvement of **5.10 percentage points** although many shots are still classified incorrectly, which suggests that the two coordinates alone provide limited information about whether a particular shot attempt will succeed in getting in the hoop. After evaluating the training recipe, I trained a fresh final model on all the supplied labeled examples and incorporated the coordinate scaling into its first layer so that the exported model accepts the original coordinate values directly.

In the [extension notebook](../Question1-Extension/submission_model.ipynb), I compared additional input combinations and training settings but the selected recipe remained the two-coordinate model with a learning rate of **0.01** and **270 epochs** as it still gave the best results out of every combination. These results describe the settings and local splits I tested, they do not establish that this recipe is the best possible model that can be built from the dataset.

Note that I had already checked this holdout during earlier experiments so the 60.36% accuracy describes performance on this local split and may differ from the result on new, unseen data.